# Teste final

**Objetivo:** medir, uma única vez, o desempenho em dados que nenhuma etapa usou.

**O que é aplicado (D20):** o modelo exatamente como foi validado. O LightGBM é treinado no treino e calibrado na 1ª metade da validação, nas versões com e sem colunas C. As filas, os custos e o limiar são os da Etapa 7.

**Regra:** depois de ver este resultado, nenhuma escolha de modelo, colunas, calibração ou custo é alterada. A 2ª metade da validação aparece ao lado para comparação.

In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
from divisao_temporal import calcular_cortes, dividir_por_tempo
from metricas import selecionar_revisadas, valor_capturado
from modelo import treinar_lightgbm

ARQUIVO = RAIZ / "data" / "processed" / "treino.parquet"
nomes = pq.ParquetFile(ARQUIVO).schema_arrow.names
COLUNAS_C = [n for n in nomes if re.fullmatch(r"C\d+", n)]
COLUNAS_ID = [n for n in nomes if re.fullmatch(r"id_\d+", n)]
COLUNAS_BASE = [
    "TransactionAmt", "tem_identidade", "ProductCD", "card4", "card6",
    "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
]

tabela = pd.read_parquet(
    ARQUIVO,
    columns=["TransactionDT", "isFraud", *[c for c in COLUNAS_BASE if c != "tem_identidade"], *COLUNAS_C, *COLUNAS_ID],
)
tabela["tem_identidade"] = tabela[COLUNAS_ID].notna().any(axis=1).astype("int8")
tabela = tabela.drop(columns=COLUNAS_ID)

corte_treino, corte_validacao = calcular_cortes(tabela["TransactionDT"])
treino, validacao, teste = dividir_por_tempo(tabela, corte_treino, corte_validacao)
del tabela

corte_metade, _ = calcular_cortes(validacao["TransactionDT"], fracao_treino=0.5, fracao_validacao=0.5)
calibra, mede, _ = dividir_por_tempo(validacao, corte_metade, validacao["TransactionDT"].max())
del _, validacao


partes = {"2ª metade da validação": mede, "Teste": teste}
pd.DataFrame(
    {
        nome: {
            "Compras": len(parte),
            "Fraudes": int(parte["isFraud"].sum()),
            "Taxa de fraude (%)": round(100 * parte["isFraud"].mean(), 2),
            "Valor total de fraude": round(float(parte.loc[parte["isFraud"] == 1, "TransactionAmt"].astype("float64").sum()), 2),
        }
        for nome, parte in partes.items()
    }
).T

,Compras,Fraudes,Taxa de fraude (%),Valor total de fraude
2ª metade da validação,59054.0,2061.0,3.49,356581.77
Teste,118108.0,4064.0,3.44,609934.34


## 1. Modelos

Mesmo treino dos notebooks 05 a 07 e mesma calibração de Platt do notebook 06. Nada aqui é ajustado com o teste.

In [2]:
def logit(p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


probabilidade = {nome: {} for nome in partes}
for versao, colunas in [("com C", [*COLUNAS_BASE, *COLUNAS_C]), ("sem C", COLUNAS_BASE)]:
    modelo, arvores = treinar_lightgbm(treino, colunas)
    platt = LogisticRegression(C=1e10).fit(logit(modelo.predict(calibra[colunas])), calibra["isFraud"])
    for nome, parte in partes.items():
        probabilidade[nome][versao] = pd.Series(
            platt.predict_proba(logit(modelo.predict(parte[colunas])))[:, 1], index=parte.index
        )
    print(f"LightGBM {versao}: {arvores} árvores")

LightGBM com C: 712 árvores


LightGBM sem C: 344 árvores


## 2. Qualidade geral

PR-AUC: quanto maior, melhor. Brier: quanto menor, melhor. Probabilidade média: numa calibração boa, fica perto da taxa real.

In [3]:
linhas = []
for nome, parte in partes.items():
    rotulos = parte["isFraud"].astype("int8")
    for versao, p in probabilidade[nome].items():
        linhas.append(
            {
                "Dados": nome,
                "Modelo": f"LightGBM {versao}",
                "PR-AUC": average_precision_score(rotulos, p),
                "Brier": brier_score_loss(rotulos, p),
                "Probabilidade média (%)": 100 * p.mean(),
                "Taxa real (%)": 100 * rotulos.mean(),
            }
        )
pd.DataFrame(linhas).set_index(["Modelo", "Dados"]).sort_index().round(4)

PR-AUC   Brier  \
Modelo         Dados                                    
LightGBM com C 2ª metade da validação  0.5461  0.0212   
               Teste                   0.4697  0.0237   
LightGBM sem C 2ª metade da validação  0.2657  0.0291   
               Teste                   0.2200  0.0297   

                                       Probabilidade média (%)  Taxa real (%)  
Modelo         Dados                                                           
LightGBM com C 2ª metade da validação                   3.5659         3.4900  
               Teste                                    3.6206         3.4409  
LightGBM sem C 2ª metade da validação                   3.5827         3.4900  
               Teste                                    3.6077         3.4409

## 3. Valor capturado por fila e capacidade

In [4]:
linhas = []
for nome, parte in partes.items():
    rotulos = parte["isFraud"].astype("int8")
    valores = parte["TransactionAmt"].astype("float64")
    filas = {
        "Regra por valor": valores,
        "LightGBM com C, P(fraude)": probabilidade[nome]["com C"],
        "LightGBM com C, P(fraude) × valor": probabilidade[nome]["com C"] * valores,
        "LightGBM sem C, P(fraude) × valor": probabilidade[nome]["sem C"] * valores,
    }
    for fila, pontuacao in filas.items():
        for capacidade in [0.005, 0.01, 0.03, 0.05]:
            revisadas = selecionar_revisadas(pontuacao, capacidade)
            linhas.append(
                {
                    "Fila": fila,
                    "Capacidade (%)": 100 * capacidade,
                    "Dados": nome,
                    "Precisão (%)": 100 * rotulos.loc[revisadas].mean(),
                    "Valor capturado (%)": valor_capturado(rotulos, pontuacao, valores, capacidade),
                }
            )
(
    pd.DataFrame(linhas)
    .pivot_table(index=["Fila", "Capacidade (%)"], columns="Dados", values=["Precisão (%)", "Valor capturado (%)"], sort=False)
    .round(2)
)

Precisão (%)  \
Dados                                            2ª metade da validação   
Fila                              Capacidade (%)                          
Regra por valor                   0.5                              3.72   
                                  1.0                              3.55   
                                  3.0                              6.88   
                                  5.0                              7.86   
LightGBM com C, P(fraude)         0.5                             88.51   
                                  1.0                             89.85   
                                  3.0                             59.65   
                                  5.0                             42.16   
LightGBM com C, P(fraude) × valor 0.5                             62.50   
                                  1.0                             61.25   
                                  3.0                             43.85   
                                  5.0                             35.02   
LightGBM sem C, P(fraude) × valor 0.5                             38.18   
                                  1.0                             31.81   
                                  3.0                             21.33   
                                  5.0                             18.02   

                                                         \
Dados                                             Teste   
Fila                              Capacidade (%)          
Regra por valor                   0.5              1.52   
                                  1.0              2.88   
                                  3.0              5.22   
                                  5.0              5.30   
LightGBM com C, P(fraude)         0.5             92.05   
                                  1.0             84.01   
                                  3.0             51.10   
                                  5.0             35.73   
LightGBM com C, P(fraude) × valor 0.5             50.08   
                                  1.0             43.65   
                                  3.0             31.72   
                                  5.0             26.77   
LightGBM sem C, P(fraude) × valor 0.5             26.57   
                                  1.0             22.17   
                                  3.0             15.89   
                                  5.0             13.92   

                                                    Valor capturado (%)         
Dados                                            2ª metade da validação  Teste  
Fila                              Capacidade (%)                                
Regra por valor                   0.5                              8.78   3.62  
                                  1.0                             12.65   9.76  
                                  3.0                             36.96  32.46  
                                  5.0                             52.06  42.97  
LightGBM com C, P(fraude)         0.5                              6.92   7.29  
                                  1.0                             17.93  14.09  
                                  3.0                             42.36  32.37  
                                  5.0                             52.82  41.49  
LightGBM com C, P(fraude) × valor 0.5                             30.79  21.35  
                                  1.0                             38.23  29.51  
                                  3.0                             57.48  49.74  
                                  5.0                             66.80  59.78  
LightGBM sem C, P(fraude) × valor 0.5                             20.72  12.62  
                                  1.0                             28.93  18.33  
                                  3.0                             44.68  35.86  
                                  5.0                

## 4. Valor líquido sob custos de revisão (premissa)

Mesmas premissas da Etapa 7 (D19). O limiar revisa toda compra em que `P(fraude) × valor` supera o custo de revisão.

In [5]:
CUSTOS = [1, 5, 10, 20, 50]
linhas = []
for nome, parte in partes.items():
    rotulos = parte["isFraud"].astype("int8")
    valores = parte["TransactionAmt"].astype("float64")
    esperado = probabilidade[nome]["com C"] * valores
    estrategias = {
        "Regra por valor, 3%": selecionar_revisadas(valores, 0.03),
        "LightGBM com C, P(fraude) × valor, 3%": selecionar_revisadas(esperado, 0.03),
    }
    for custo in CUSTOS:
        for estrategia, revisadas in [*estrategias.items(), ("LightGBM com C, limiar", esperado.index[esperado > custo])]:
            capturado = float(valores.loc[revisadas][rotulos.loc[revisadas] == 1].sum())
            linhas.append(
                {
                    "Custo por revisão": custo,
                    "Estratégia": estrategia,
                    "Dados": nome,
                    "Revisões": len(revisadas),
                    "Valor líquido": capturado - custo * len(revisadas),
                }
            )
(
    pd.DataFrame(linhas)
    .pivot_table(index=["Custo por revisão", "Estratégia"], columns="Dados", values=["Revisões", "Valor líquido"], sort=False)
    .round(0)
)

Revisões  \
Dados                                                   2ª metade da validação   
Custo por revisão Estratégia                                                     
1                 Regra por valor, 3%                                   1772.0   
                  LightGBM com C, P(fraude) × valor, 3%                 1772.0   
                  LightGBM com C, limiar                               23764.0   
5                 Regra por valor, 3%                                   1772.0   
                  LightGBM com C, P(fraude) × valor, 3%                 1772.0   
                  LightGBM com C, limiar                                8660.0   
10                Regra por valor, 3%                                   1772.0   
                  LightGBM com C, P(fraude) × valor, 3%                 1772.0   
                  LightGBM com C, limiar                                4943.0   
20                Regra por valor, 3%                                   1772.0   
                  LightGBM com C, P(fraude) × valor, 3%                 1772.0   
                  LightGBM com C, limiar                                2692.0   
50                Regra por valor, 3%                                   1772.0   
                  LightGBM com C, P(fraude) × valor, 3%                 1772.0   
                  LightGBM com C, limiar                                 963.0   

                                                                  \
Dados                                                      Teste   
Custo por revisão Estratégia                                       
1                 Regra por valor, 3%                     3544.0   
                  LightGBM com C, P(fraude) × valor, 3%   3544.0   
                  LightGBM com C, limiar                 49668.0   
5                 Regra por valor, 3%                     3544.0   
                  LightGBM com C, P(fraude) × valor, 3%   3544.0   
                  LightGBM com C, limiar                 18330.0   
10                Regra por valor, 3%                     3544.0   
                  LightGBM com C, P(fraude) × valor, 3%   3544.0   
                  LightGBM com C, limiar                 10712.0   
20                Regra por valor, 3%                     3544.0   
                  LightGBM com C, P(fraude) × valor, 3%   3544.0   
                  LightGBM com C, limiar                  5739.0   
50                Regra por valor, 3%                     3544.0   
                  LightGBM com C, P(fraude) × valor, 3%   3544.0   
                  LightGBM com C, limiar                  2070.0   

                                                                 Valor líquido  \
Dados                                                   2ª metade da validação   
Custo por revisão Estratégia                                                     
1                 Regra por valor, 3%                                 130023.0   
                  LightGBM com C, P(fraude) × valor, 3%               203191.0   
                  LightGBM com C, limiar                              322620.0   
5                 Regra por valor, 3%                                 122935.0   
                  LightGBM com C, P(fraude) × valor, 3%               196103.0   
                  LightGBM com C, limiar                              262596.0   
10                Regra por valor, 3%                                 114075.0   
                  LightGBM com C, P(fraude) × valor, 3%               187243.0   
                  LightGBM com C, limiar                              222615.0   
20                Regra por valor, 3%                                  96355.0   
                  LightGBM com C, P(fraude) × valor, 3%               169523.0   
                  LightGBM com C, limiar                              180293.0   
50                Regra por valor, 3%                                  43195.0   
                  LightGBM com C, P(fraude) × valor, 3% 

## Conclusões do teste final

- O desempenho caiu do período de validação para o teste. A PR-AUC do LightGBM com C foi de 0,5461 na 2ª metade da validação para 0,4697 no teste. Sem C, foi de 0,2657 para 0,2200.
- A queda acompanha a distância no tempo. Com C, a PR-AUC foi de 0,5742 na validação inteira para 0,5461 na 2ª metade e 0,4697 no teste. O treino termina no dia 101 e o teste começa depois do dia 141. A regra por valor, que não treina nada, também caiu: o valor capturado em 3% foi de 36,96% para 32,46%. Isso indica que os dados mudaram com o tempo, e não só que o modelo errou. Algum ajuste excessivo aos dados de validação pode explicar uma parte da queda, mas este teste sozinho não separa as duas causas.
- A conclusão principal da Etapa 7 continua valendo. Em 3% de capacidade, a fila por `P(fraude) × valor` captura 49,74% do valor de fraude, contra 32,46% da regra por valor e 32,37% da fila por `P(fraude)`. O ganho sobre a regra por valor caiu de 20,52 para 17,28 pontos, mas continua grande. Sem colunas C, a fila por valor esperado também supera a regra (35,86% contra 32,46%).
- A fila por `P(fraude)` sozinha perdeu a vantagem sobre a regra por valor no teste (32,37% contra 32,46%). Ordenar só pela chance de fraude não basta para capturar valor.
- A calibração continuou próxima da realidade: probabilidade média de 3,62% com C, para uma taxa real de 3,44%.
- O limiar também manteve o maior valor líquido em todos os custos testados, mas revisa muito mais compras quando o custo é baixo: 49.668 com custo 1 e 10.712 com custo 10, num teste de 118.108 compras.
- Nada foi ajustado depois de ver o teste (D20). Ajustar agora transformaria o teste em mais uma validação, e o resultado deixaria de medir o desempenho em dados novos.
- **Recomendação para produção:** treinar de novo todo mês com os dados novos que já tiverem rótulo, refazer a calibração e acompanhar mês a mês a precisão e o valor de fraude capturado. Isso corrige o envelhecimento do modelo e mostra, com o tempo, quanto ele perde por mês sem atualização. A frequência mensal é uma proposta, não foi testada aqui. Como o rótulo só chega depois do chargeback, cada atualização usa dados com algumas semanas de atraso.
- **Melhorias futuras:** qualquer mudança no modelo motivada por este resultado precisa ser medida em dados novos, e não neste teste. Uma forma é simular a passagem do tempo dentro do treino e da validação: treinar até um dia, medir nas semanas seguintes e repetir com vários dias.
- **Limites:** é um único período de teste. As colunas C não podem ser auditadas. O custo de revisão é uma premissa. O teste tem o dobro de compras da 2ª metade da validação, então os valores absolutos não são comparáveis entre as duas colunas, só as porcentagens.